In [ ]:
import sys
import os
import re
import requests
import pandas as pd

sys.path.insert(0, '/Users/minhee/capstone-project')
os.environ['DJANGO_SETTINGS_MODULE'] = 'capstone_project.settings'
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'

import django
django.setup()

from artifacts.models import Artifact
from sentence_transformers import SentenceTransformer
print("환경 설정 완료!")

환경 설정 완료!


In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("모델 로드 완료!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6890.84it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 로드 완료!


In [ ]:
resp = requests.get(
    "https://openaccess-api.clevelandart.org/api/artworks/",
    params={"has_image": 1, "currently_on_view": 1, "limit": 1}
)
total = resp.json()['info']['total']
print(f"전시 중 + 이미지 있는 유물 수: {total}")

전시 중 + 이미지 있는 유물 수: 6243


In [24]:
# 전시 중인 cleveland_id 목록 수집
on_view_ids = set()
for i in range(0, 7000, 100):
    data = fetch_cma_artifacts(limit=100, skip=i)
    for aw in data:
        on_view_ids.add(aw['id'])
    print(f"{i} 완료, 현재 수집: {len(on_view_ids)}")

print(f"전시 중 유물 수: {len(on_view_ids)}")

# 전시 중 아닌 것 is_active=False 처리
updated = Artifact.objects.exclude(cleveland_id__in=on_view_ids).update(is_active=False)
print(f"비활성화된 유물 수: {updated}")

# 확인
print(f"활성 유물 수: {Artifact.objects.filter(is_active=True).count()}")

0 완료, 현재 수집: 100
100 완료, 현재 수집: 200
200 완료, 현재 수집: 300
300 완료, 현재 수집: 400
400 완료, 현재 수집: 500
500 완료, 현재 수집: 600
600 완료, 현재 수집: 700
700 완료, 현재 수집: 800
800 완료, 현재 수집: 900
900 완료, 현재 수집: 1000
1000 완료, 현재 수집: 1100
1100 완료, 현재 수집: 1200
1200 완료, 현재 수집: 1300
1300 완료, 현재 수집: 1400
1400 완료, 현재 수집: 1498
1500 완료, 현재 수집: 1598
1600 완료, 현재 수집: 1698
1700 완료, 현재 수집: 1798
1800 완료, 현재 수집: 1898
1900 완료, 현재 수집: 1998
2000 완료, 현재 수집: 2098
2100 완료, 현재 수집: 2198
2200 완료, 현재 수집: 2298
2300 완료, 현재 수집: 2398
2400 완료, 현재 수집: 2498
2500 완료, 현재 수집: 2598
2600 완료, 현재 수집: 2698
2700 완료, 현재 수집: 2798
2800 완료, 현재 수집: 2898
2900 완료, 현재 수집: 2998
3000 완료, 현재 수집: 3097
3100 완료, 현재 수집: 3197
3200 완료, 현재 수집: 3297
3300 완료, 현재 수집: 3397
3400 완료, 현재 수집: 3497
3500 완료, 현재 수집: 3597
3600 완료, 현재 수집: 3697
3700 완료, 현재 수집: 3797
3800 완료, 현재 수집: 3895
3900 완료, 현재 수집: 3995
4000 완료, 현재 수집: 4095
4100 완료, 현재 수집: 4195
4200 완료, 현재 수집: 4292
4300 완료, 현재 수집: 4390
4400 완료, 현재 수집: 4490
4500 완료, 현재 수집: 4585
4600 완료, 현재 수집: 4685
4700 완료, 현재 수집: 4785
4800 완료, 현재 수

In [ ]:
def make_embedding_text(aw):
    culture = aw.get('culture', '')
    if isinstance(culture, list):
        culture = ', '.join(culture)

    parts = [
        aw.get('title', ''),
        aw.get('type', ''),
        aw.get('department', ''),
        aw.get('collection', ''),
        aw.get('technique', ''),
        culture,
        aw.get('description', '') or '',
        aw.get('did_you_know', '') or '',
    ]
    return ' '.join([p for p in parts if p])

In [ ]:
def sync_artifacts(limit=100, skip=0):
    artworks = fetch_cma_artifacts(limit=limit, skip=skip)
    created, updated = 0, 0

    for aw in artworks:
        cleveland_id = aw.get('id')
        if not cleveland_id:
            continue

        # 이미지 URL
        images = aw.get('images') or {}
        image_url = ''
        if images.get('web'):
            image_url = images['web'].get('url', '')

        # 이미지 없으면 스킵
        if not image_url:
            continue

        # 전시 위치 없으면 스킵
        if not aw.get('current_location'):
            continue

        # culture 처리
        culture = aw.get('culture', [])
        if isinstance(culture, str):
            culture = [culture] if culture else []

        # 임베딩 생성
        embedding_text = make_embedding_text(aw)
        embedding_vector = model.encode(embedding_text).tolist() if embedding_text else None

        obj, is_created = Artifact.objects.update_or_create(
            cleveland_id=cleveland_id,
            defaults={
                'title': aw.get('title', ''),
                'type': aw.get('type', ''),
                'department': aw.get('department', ''),
                'collection': aw.get('collection', ''),
                'technique': aw.get('technique', ''),
                'culture': culture,
                'creation_date_earliest': aw.get('creation_date_earliest'),
                'creation_date_latest': aw.get('creation_date_latest'),
                'creation_date': aw.get('creation_date', ''),
                'current_location': aw.get('current_location', '') or '',
                'image_url': image_url,
                'description': aw.get('description', '') or '',
                'did_you_know': aw.get('did_you_know', '') or '',
                'accession_number': aw.get('accession_number', ''),
                'tombstone': aw.get('tombstone', '') or '',
                'measurements': aw.get('measurements', '') or '',
                'share_license_status': aw.get('share_license_status', 'CC0'),
                'embedding_text': embedding_text,
                'embedding_vector': embedding_vector,
                'is_active': True,
            }
        )

        if is_created:
            created += 1
        else:
            updated += 1

        if (created + updated) % 50 == 0:
            print(f"진행 중... 생성: {created}, 업데이트: {updated}")

    print(f"완료 - 생성: {created}, 업데이트: {updated}")

In [21]:
for i in range(0, total, 100):
    print(f"skip={i} 처리 중...")
    sync_artifacts(limit=100, skip=i)

skip=0 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=100 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=200 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=300 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=400 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=500 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=600 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=700 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=800 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=900 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=1000 처리 중...
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 100
완료 - 생성: 0, 업데이트: 100
skip=1100 처리 중...
진행 중... 생성: 0, 

In [7]:
# 5. DB 저장 + 임베딩 생성
def sync_artifacts(limit=100, skip=0):
    artworks = fetch_cma_artifacts(limit=limit, skip=skip)
    created, updated = 0, 0

    for aw in artworks:
        cleveland_id = aw.get('id')
        if not cleveland_id:
            continue

        # 이미지 URL
        images = aw.get('images') or {}
        image_url = ''
        if images.get('web'):
            image_url = images['web'].get('url', '')

        # culture 처리 (list로 저장)
        culture = aw.get('culture', [])
        if isinstance(culture, str):
            culture = [culture] if culture else []

        # 임베딩 텍스트 + 벡터 생성
        embedding_text = make_embedding_text(aw)
        embedding_vector = model.encode(embedding_text).tolist() if embedding_text else None

        obj, is_created = Artifact.objects.update_or_create(
            cleveland_id=cleveland_id,
            defaults={
                'title': aw.get('title', ''),
                'type': aw.get('type', ''),
                'department': aw.get('department', ''),
                'collection': aw.get('collection', ''),
                'technique': aw.get('technique', ''),
                'culture': culture,
                'creation_date_earliest': aw.get('creation_date_earliest'),
                'creation_date_latest': aw.get('creation_date_latest'),
                'creation_date': aw.get('creation_date', ''),
                'current_location': aw.get('current_location', '') or '',
                'image_url': image_url,
                'description': aw.get('description', '') or '',
                'did_you_know': aw.get('did_you_know', '') or '',
                'accession_number': aw.get('accession_number', ''),
                'tombstone': aw.get('tombstone', '') or '',
                'measurements': aw.get('measurements', '') or '',
                'share_license_status': aw.get('share_license_status', 'CC0'),
                'embedding_text': embedding_text,
                'embedding_vector': embedding_vector,
                'is_active': True,
            }
        )

        if is_created:
            created += 1
        else:
            updated += 1

        if (created + updated) % 10 == 0:
            print(f"진행 중... 생성: {created}, 업데이트: {updated}")

    print(f"완료 - 생성: {created}, 업데이트: {updated}")

sync_artifacts(limit=7000, skip=0)

진행 중... 생성: 0, 업데이트: 10
진행 중... 생성: 0, 업데이트: 20
진행 중... 생성: 0, 업데이트: 30
진행 중... 생성: 0, 업데이트: 40
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 60
진행 중... 생성: 0, 업데이트: 70
진행 중... 생성: 0, 업데이트: 80
진행 중... 생성: 0, 업데이트: 90
진행 중... 생성: 0, 업데이트: 100
진행 중... 생성: 0, 업데이트: 110
진행 중... 생성: 0, 업데이트: 120
진행 중... 생성: 0, 업데이트: 130
진행 중... 생성: 0, 업데이트: 140
진행 중... 생성: 0, 업데이트: 150
진행 중... 생성: 0, 업데이트: 160
진행 중... 생성: 0, 업데이트: 170
진행 중... 생성: 0, 업데이트: 180
진행 중... 생성: 0, 업데이트: 190
진행 중... 생성: 0, 업데이트: 200
진행 중... 생성: 0, 업데이트: 210
진행 중... 생성: 0, 업데이트: 220
진행 중... 생성: 0, 업데이트: 230
진행 중... 생성: 0, 업데이트: 240
진행 중... 생성: 0, 업데이트: 250
진행 중... 생성: 0, 업데이트: 260
진행 중... 생성: 0, 업데이트: 270
진행 중... 생성: 0, 업데이트: 280
진행 중... 생성: 0, 업데이트: 290
진행 중... 생성: 0, 업데이트: 300
진행 중... 생성: 0, 업데이트: 310
진행 중... 생성: 0, 업데이트: 320
진행 중... 생성: 0, 업데이트: 330
진행 중... 생성: 0, 업데이트: 340
진행 중... 생성: 0, 업데이트: 350
진행 중... 생성: 0, 업데이트: 360
진행 중... 생성: 0, 업데이트: 370
진행 중... 생성: 0, 업데이트: 380
진행 중... 생성: 0, 업데이트: 390
진행 중... 생성: 0, 업데이트: 400
진행 중... 생

In [8]:
# 6. 확인
total = Artifact.objects.filter(embedding_vector__isnull=False, is_active=True).count()
print(f"임베딩 완료된 유물 수: {total}")

# 샘플 확인
sample = Artifact.objects.filter(embedding_vector__isnull=False).first()
print(f"샘플 유물: {sample.title}")
print(f"임베딩 벡터 길이: {len(sample.embedding_vector)}")

임베딩 완료된 유물 수: 33710
샘플 유물: Leda and the Swan
임베딩 벡터 길이: 384


In [23]:
ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]')

def clean_text(val):
    if isinstance(val, str):
        return ILLEGAL_CHARACTERS_RE.sub('', val)
    return val

df = pd.DataFrame(list(Artifact.objects.filter(is_active=True).values()))
df = df.map(clean_text)  # applymap → map
df['culture'] = df['culture'].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))
df.drop(columns=['embedding_vector'], inplace=True)

df.to_excel('/Users/minhee/Desktop/artifacts_db.xlsx', index=False)
print(f"완료! {len(df)}개 유물 저장됨")

완료! 41286개 유물 저장됨
